In [1]:
from ecspr.model.graph import AtomGraph, Terminal, solve

In [2]:

# One constructed edge A -> B, bypassing the bake/pairs pipeline entirely.
# gp/gm are the forward/backward conductances directly (gp == gm is the
# undirected/symmetric limit; set gm < gp to make the edge a rectifier).
A = ("A", 0)
B = ("B", 0)

graph = AtomGraph.from_edge_records(
    [(A, B, 1.0, 1.0)],  # (tail, head, gp, gm)
    meta={"note": "single constructed edge A->B"},
)

source = Terminal.of_nodes("A", [A])
sink = Terminal.of_nodes("B", [B])
sol = solve(graph, source, sink)

sol

Solution(total=1, A->B, converged=True)

In [3]:
print("total (effective conductance A->B):", sol.total)
print("converged:", sol.converged)
print("current at A:", sol.current(A), " current at B:", sol.current(B))
print("voltage at A:", sol.voltage(A), " voltage at B:", sol.voltage(B))

total (effective conductance A->B): 1.0
converged: True
current at A: -1.0  current at B: 1.0
voltage at A: 1.0  voltage at B: 0.0


## Two-point probe, e_coli_epi300: before/after a `serA` knockout

`serA` (Phosphoglycerate dehydrogenase, `PGCD` / `MNXR102527`) is the first committed step
of serine biosynthesis from 3-phosphoglycerate. A knockout is not an argument to the solver —
it is a **mask over the rows of the GPR table** that drops `serA`'s evidence, which yields a
smaller reaction-weight set and therefore a different network. The two-point conductance
D-glucose -> L-serine is then just two independent solves, compared by the caller.

In [4]:
import polars as pl

GPR_PATH = "/home/tony/agentic_workspace/projects/metasmith/fabfos/bench-eydallin/data/fabfos/runs/e_coli_epi300/gpr/gpr_gem.parquet"

gpr_pl = pl.read_parquet(GPR_PATH)

# find serA: search every string column rather than assuming which one names it
str_cols = [c for c, dt in zip(gpr_pl.columns, gpr_pl.dtypes) if dt == pl.Utf8]
hit_mask = pl.any_horizontal(
    [pl.col(c).str.to_lowercase() == "sera" for c in str_cols]
)
gpr_pl.filter(hit_mask)

source,orf,channel,mnxr,intermediate_id,intermediate_name,raw_score,score_kind,projection_via,evidence_quality,lane_set,build_id,host,unit_id,feature_kind,feature_name,gpr_rule,in_atom_universe
str,str,str,str,str,str,f32,str,str,str,str,str,str,str,str,str,str,bool
"""iECDH10B_1368""","""ECDH10B_3088""","""gem_gpr""","""MNXR102527""","""PGCD""","""Phosphoglycerate dehydrogenase""",1.0,"""presence""","""bigg""","""unknown""","""curated""","""gem_iECDH10B_1368""","""e_coli_epi300""","""iECDH10B_1368""","""gem_gene""","""serA""","""ECDH10B_3088""",true


In [5]:
from ecspr.model.gpr import condition_weights

# serA is one gene row nominating MNXR102527 (PGCD, phosphoglycerate dehydrogenase).
# Dropping its row is the whole knockout: fewer rows -> smaller/absent weight for that
# reaction -> a different network, never a special-cased "knockout" argument to the solver.
# pool=True runs the log-odds pooling (off by default) so E_r stays a bounded probability.
SERA_MNXR = gpr_pl.filter(hit_mask)["mnxr"].item(0)
print("serA reaction:", SERA_MNXR)

# condition_weights is pandas-native, so cross the boundary here.
gpr = gpr_pl.to_pandas()
w_wildtype, cov_wildtype = condition_weights(gpr, pool=True)
w_knockout, cov_knockout = condition_weights(gpr, drop_column="feature_name", drop_values=["serA"], pool=True)

print("wildtype coverage:", cov_wildtype)
print("knockout coverage:", cov_knockout)
print(f"E_r[{SERA_MNXR}] wildtype:", w_wildtype.get(SERA_MNXR),
      " knockout:", w_knockout.get(SERA_MNXR))

serA reaction: MNXR102527
wildtype coverage: {'n_rows': 4814, 'n_units': 1, 'n_reactions': 2290, 'n_rows_in_atom_universe': 4136}
knockout coverage: {'n_rows': 4813, 'n_units': 1, 'n_reactions': 2289, 'n_rows_in_atom_universe': 4135}
E_r[MNXR102527] wildtype: 0.1172932697601497  knockout: None


In [6]:
from ecspr.bake.encoding import load_vocab, read_identity, load_direction, compile_atom_graph

BAKE_DIR = "/home/tony/agentic_workspace/projects/metasmith/fabfos/bench-eydallin/data/fabfos/processed/metabolism_bake"

vocab = load_vocab(f"{BAKE_DIR}/vocab.parquet")
ident = read_identity(f"{BAKE_DIR}/atom_pairs.parquet")
pairs = pl.read_parquet(f"{BAKE_DIR}/atom_pairs.parquet").to_pandas()  # compile_atom_graph is pandas-native
direction = load_direction(f"{BAKE_DIR}/direction.parquet")

graph_wildtype = compile_atom_graph("C", w_wildtype, ident=ident, vocab=vocab,
                                    pairs=pairs, direction=direction)
graph_knockout = compile_atom_graph("C", w_knockout, ident=ident, vocab=vocab,
                                    pairs=pairs, direction=direction)

print("wildtype graph:", graph_wildtype.n, "nodes,", graph_wildtype.m, "edges")
print("knockout graph:", graph_knockout.n, "nodes,", graph_knockout.m, "edges")

wildtype graph: 21198 nodes, 31850 edges
knockout graph: 21198 nodes, 31847 edges


In [7]:
GLUCOSE = "MNXM1364061"   # D-glucose
SERINE = "MNXM737787"     # L-serine

results = {}
for label, g in [("wildtype", graph_wildtype), ("serA_knockout", graph_knockout)]:
    source = Terminal.merge(g, [GLUCOSE])
    sink = Terminal.merge(g, [SERINE])
    sol = solve(g, source, sink)
    results[label] = sol
    print(f"{label:14s} total={sol.total:.10f}  converged={sol.converged}  "
          f"missing_source={source.missing}  missing_sink={sink.missing}")

delta = results["serA_knockout"].total - results["wildtype"].total
pct = 100 * delta / results["wildtype"].total
print(f"\nglucose -> serine conductance change from the serA knockout: {delta:.3e} ({pct:.4f}%)")

wildtype       total=0.0645533822  converged=True  missing_source=()  missing_sink=()


serA_knockout  total=0.0645525393  converged=True  missing_source=()  missing_sink=()

glucose -> serine conductance change from the serA knockout: -8.429e-07 (-0.0013%)
